<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/04_yoctoGPT_walden_char.ipynb)


# yoctoGPT — Walden Character-Level Demo

## _Training a Tiny GPT on Thoreau’s Walden (CPU vs GPU)_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Train a tiny character-level GPT on Thoreau’s *Walden*
  and compare a quick CPU run with a GPU-accelerated run.
- **Hardware**: Designed for Google Colab (T4 default). GPU is auto-detected.
- **Approach**: We use `yoctoGPT`, a minimal PyTorch implementation.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the GPU comparison.

### Roadmap

1. **Setup**: Clone `yoctoGPT` and download *Walden*.
2. **Data**: Prepare a clean character-level dataset.
3. **CPU Training**: A very small model trained for 500 iters on CPU.
4. **GPU Training**: Use the `char_small` profile from
   `gpu_configs.yml` on the detected GPU.
5. **Sampling**: Generate text from both checkpoints and compare.

### Setup: Clone Repository and Download Text

We install dependencies, clone `yoctoGPT`, and fetch *Walden* from
`hilpisch.com`.

In [ ]:
#@title Check GPU and install dependencies
!nvidia-smi
!pip -q install tokenizers tqdm textstat pyyaml

import os
import pathlib
import subprocess
import urllib.request

repo_root = pathlib.Path("/content/yoctoGPT")
if not repo_root.exists():
    print("Cloning yoctoGPT repository...")
    subprocess.run([
        "git", "clone",
        "https://github.com/yhilpisch/yoctoGPT.git",
        str(repo_root),
    ])
os.chdir(repo_root)
print(f"Working directory: {os.getcwd()}")

walden_path = pathlib.Path("data/walden.txt")
if not walden_path.exists():
    print("Downloading Walden...")
    urllib.request.urlretrieve(
        "https://hilpisch.com/walden.txt", str(walden_path)
    )
print(f"Corpus size: {walden_path.stat().st_size / 1e6:.2f} MB")

In [ ]:
#@title Detect GPU and load training profile
import subprocess, yaml

_out = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name",
     "--format=csv,noheader"],
    text=True,
).strip()

if "T4" in _out:
    GPU_KEY = "t4"
elif "L4" in _out:
    GPU_KEY = "l4"
elif "A100" in _out:
    GPU_KEY = "a100"
elif "RTX PRO 6000" in _out:
    GPU_KEY = "g4"
elif "H100" in _out:
    GPU_KEY = "h100"
else:
    GPU_KEY = "t4"

GPU_CFG = yaml.safe_load(
    open("notebooks/gpu_configs.yml")
)[GPU_KEY]
AMP_DTYPE = GPU_CFG["amp_dtype"]
print(f"GPU: {_out} -> {GPU_KEY}, amp_dtype: {AMP_DTYPE}")

### Data Preparation

We convert the raw text into a character-level vocabulary: every
letter, digit, and punctuation mark gets a unique integer ID.

In [ ]:
#@title Prepare character-level dataset
!python -m scripts.prepare_char_data \
    --text_path data/walden.txt \
    --out_dir data/char_walden \
    --sanitize_chars ascii \
    --no_punctuation \
    --keep_period \
    --collapse_whitespace \
    --lowercase

## Part A: CPU Training — Minimal Model

A deliberately tiny run that finishes in seconds on CPU.
This proves the pipeline works even without a GPU.

In [ ]:
#@title Train minimal model on CPU
import time

start = time.perf_counter()
!python -m yoctoGPT.train \
    --mode char \
    --data_dir data/char_walden \
    --ckpt_dir checkpoints/cpu_walden \
    --device cpu \
    --n_layer 2 \
    --n_head 2 \
    --n_embd 64 \
    --block_size 64 \
    --batch_size 16 \
    --max_iters 500 \
    --lr 1e-3 \
    --eval_interval 100
cpu_time = time.perf_counter() - start
print(f"\nCPU training time: {cpu_time:.1f} s")

In [ ]:
#@title Sample from the CPU checkpoint
prompt = "the mass of men lead lives of "
cpu_out = !python -m yoctoGPT.sampler \
    --mode char \
    --ckpt checkpoints/cpu_walden/latest.pt \
    --vocab_path data/char_walden/vocab.json \
    --prompt "{prompt}" \
    --max_new_tokens 200
cpu_text = "\n".join(cpu_out)
print(cpu_text)

## Part B: GPU Training — Scaled Model

We reuse the same corpus but train a larger model on the free Colab
GPU. We start from the `char_small` profile but bump the architecture
to 4 layers / 256 dims for noticeably better output in ~60 s.
Even 3 000 iterations on a T4 finish in about a minute.

In [ ]:
#@title Load profile and bump capacity
P = GPU_CFG["char_small"]
# Profile block_size / batch_size / amp_dtype stay as-is
block_size = P["block_size"]
batch_size = P["batch_size"]
# Bump architecture for better quality on this corpus
n_layer, n_head, n_embd = 4, 8, 256
print(
    f"walden_gpu: n_layer={n_layer}, n_head={n_head}, "
    f"n_embd={n_embd}, block_size={block_size}, "
    f"batch_size={batch_size}"
)

In [ ]:
#@title Train on GPU (char_small profile)
start = time.perf_counter()
!python -m yoctoGPT.train \
    --mode char \
    --data_dir data/char_walden \
    --ckpt_dir checkpoints/gpu_walden \
    --n_layer {n_layer} \
    --n_head {n_head} \
    --n_embd {n_embd} \
    --block_size {block_size} \
    --batch_size {batch_size} \
    --max_iters 3000 \
    --lr 1e-3 \
    --eval_interval 500 \
    --amp --amp_dtype {AMP_DTYPE}
gpu_time = time.perf_counter() - start
print(f"\nGPU training time: {gpu_time:.1f} s")

In [ ]:
#@title Sample from the GPU checkpoint
gpu_out = !python -m yoctoGPT.sampler \
    --mode char \
    --ckpt checkpoints/gpu_walden/latest.pt \
    --vocab_path data/char_walden/vocab.json \
    --prompt "{prompt}" \
    --max_new_tokens 200
gpu_text = "\n".join(gpu_out)
print(gpu_text)

### Side-by-Side Comparison

The same prompt fed to both checkpoints. The GPU model has more
parameters and trained longer, so its output should look a little
less random.

In [ ]:
#@title Compare outputs
print("=" * 60)
print("CPU model  (2 layers, 64 dim, 500 iters)")
print("=" * 60)
print(cpu_text)
print("\n" + "=" * 60)
print(f"GPU model  ({n_layer} layers, {n_embd} dim, 3000 iters)")
print("=" * 60)
print(gpu_text)
print("\n" + "-" * 60)
print(f"CPU time: {cpu_time:.1f} s  |  GPU time: {gpu_time:.1f} s")

### Text Quality Scorecard

We score both CPU and GPU outputs against the original *Walden*
corpus using readability, diversity, and distributional metrics.
Lower Char-KL means the generated text is closer to Thoreau’s
character distribution.

In [ ]:
#@title Score generated text
from yoctoGPT.text_metrics import score_text, print_scorecard

reference = pathlib.Path("data/walden.txt").read_text(
    encoding="utf-8"
)

print("CPU model  (2 layers, 64 dim, 500 iters)")
print("-" * 50)
cpu_card = score_text(cpu_text, reference_text=reference)
print_scorecard(cpu_card)

print("\nGPU model  (4 layers, 256 dim, 3000 iters)")
print("-" * 50)
gpu_card = score_text(gpu_text, reference_text=reference)
print_scorecard(gpu_card)

print("\n=> Lower Char-KL is closer to Walden.")


### Exercises

1. **Even longer training**: Resume the GPU checkpoint with
   `--resume checkpoints/gpu_walden/latest.pt --max_iters 3000`.
   Does the text improve further?
2. **Bigger context**: Increase `block_size` to 256 for the GPU run.
   How much VRAM does it use?
3. **Token mode**: Use `scripts.prepare_tokenizer` to create a BPE
   vocabulary and train a token-level model on *Walden*.
4. **Custom prompt**: Try `"i went to the woods because"` and compare
   CPU vs GPU generation quality.
5. **Deeper model**: Bump `n_layer` to 6 and `n_embd` to 384.
   How does the loss curve look after 3000 iterations?

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
